<a href="https://colab.research.google.com/github/WhoisMonesh/Colab-Mega-Downloader/blob/main/Colab-Mega-Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mega Downloader

Download files from Mega.nz links directly to Google Drive.


### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies

In [0]:
!pip install mega.py -q
print('Dependencies installed.')

### 3. Configuration

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/MegaDownloader/'  # local temp dir
DRIVE_PATH = '/content/drive/My Drive/MegaDownloader/'  # final destination

MEGA_LINK = ""  # Mega.nz file/folder link
MEGA_EMAIL = ""  # Mega account email (optional)
MEGA_PASSWORD = ""  # Mega account password

KEEP_ALIVE = True  # prevent Colab timeout

# === END CONFIGURATION ===

### 4. Download

In [0]:
import os, time, shutil
from google.colab import files

def format_bytes(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def get_all_files(root):
    result = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            result.append(os.path.join(dirpath, f))
    return result

def download_with_progress(url, fpath, progress_display, label):
    import requests
    r = requests.get(url, stream=True)
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(fpath, 'wb') as f:
        for chunk in r.iter_content(8192):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = int(downloaded * 100 / total)
                    bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                    progress_display.update(HTML(f'<pre>Downloading: |{bar}| {pct}% | {label} ({format_bytes(downloaded)}/{format_bytes(total)})</pre>'))
    return downloaded

def download_mega_node(mega, node, save_path, progress_display, depth=0):
    if 'a' in node and node['a'].get('n'):
        # file node
        name = node['a']['n']
        fpath = os.path.join(save_path, name)
        dl_url = mega._get_file_link(node)
        sz = download_with_progress(dl_url, fpath, progress_display, name)
        return 1
    elif 't' in node and node['t'] == 1:
        # folder node
        folder_name = node['a'].get('n', 'folder')
        folder_path = os.path.join(save_path, folder_name)
        os.makedirs(folder_path, exist_ok=True)
        count = 0
        for child in mega._api_get_nodes(node):
            count += download_mega_node(mega, child, folder_path, progress_display, depth + 1)
        return count
    return 0

def main():
    from IPython.display import display, HTML, Javascript

    if not MEGA_LINK:
        print('ERROR: Set MEGA_LINK in config cell')
        return

    if KEEP_ALIVE:
        display(Javascript('''
            function keepAlive(){
                var btn=document.querySelector("colab-connect-button");
                if(btn)btn.click()
            }
            setInterval(keepAlive,120000);
        '''))
        print('Keep-alive active')

    os.makedirs(SAVE_PATH, exist_ok=True)
    print(f'Save path: {SAVE_PATH} (local)')

    begin = time.time()
    progress_display = display(HTML('<pre>Starting...</pre>'), display_id='dl-progress')

    from mega import Mega
    print('Connecting to Mega...')
    mega = Mega()
    if MEGA_EMAIL and MEGA_PASSWORD:
        mega.login(MEGA_EMAIL, MEGA_PASSWORD)
        print('Logged in with account')
    else:
        mega.login()
        print('Logged in anonymously (limited transfer)')

    print(f'Parsing: {MEGA_LINK}')
    node = mega.find_url(MEGA_LINK)
    if not node:
        print('ERROR: Could not find Mega link')
        return

    total_files = download_mega_node(mega, node, SAVE_PATH, progress_display)

    end = time.time()
    elapsed = int(end - begin)
    print()
    print('=' * 50)
    print('COMPLETE')
    print(f'Files: {total_files}')
    print(f'Elapsed: {elapsed // 60}m {elapsed % 60}s')
    print(f'Saved locally: {SAVE_PATH}')

    items = get_all_files(SAVE_PATH)
    if not items:
        print('No files downloaded.')
        return
    total_sz = sum(os.path.getsize(f) for f in items)
    print(f'Downloaded {len(items)} files ({format_bytes(total_sz)})')

    if len(items) > 1:
        import zipfile
        processed = 0
        zpath = SAVE_PATH.rstrip('/').rstrip('\\') + '.zip'
        print(f'\nZipping {len(items)} files...')
        with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fp in items:
                arcname = os.path.relpath(fp, SAVE_PATH)
                zf.write(fp, arcname)
                processed += os.path.getsize(fp)
                pct = int(processed * 100 / total_sz) if total_sz else 0
                bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                progress_display.update(HTML(f'<pre>Zipping: |{bar}| {pct}% | {arcname}</pre>'))
        zip_name = os.path.basename(zpath)
        zip_size = os.path.getsize(zpath)
        print(f'Zip: {zip_name} ({format_bytes(zip_size)})')

    print(f'\nMoving to Drive...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for f in os.listdir(SAVE_PATH):
        shutil.move(os.path.join(SAVE_PATH, f), os.path.join(DRIVE_PATH, f))
    print(f'Final: {DRIVE_PATH}')

    if len(items) > 1:
        drive_zip = os.path.join(DRIVE_PATH, zip_name)
        if os.path.exists(drive_zip):
            display(HTML(f'<p>Zip saved to Drive:<br><a href="{drive_zip}" download>{zip_name} ({format_bytes(zip_size)})</a></p>'))
            if zip_size < 500 * 1024 * 1024:
                files.download(drive_zip)
            else:
                print(f'Large file ({format_bytes(zip_size)}) — open Drive to download.')

    print('=' * 50)

if __name__ == '__main__':
    main()
